# Phase 1 & 2: Setup Técnico + Análise Piloto

**Interpretabilidade de Modelos de Linguagem em Português via Sparse Autoencoders**

---

**Fase 1** — Setup Técnico
- Instalar dependências (SAELens, TransformerLens)
- Carregar Gemma 2 2B + SAE do Gemma Scope (layer 13, 16k features)
- Teste básico

**Fase 2** — Análise Piloto
- Processar ~1M tokens PT + EN (Wikipedia)
- Calcular Language Specificity Index (LSI)
- Identificar top-10 features PT-específicas
- Gerar visualizações

**Pré-requisitos:**
- GPU com ≥16GB VRAM (T4 no Colab funciona)
- [Aceitar licença do Gemma 2](https://huggingface.co/google/gemma-2-2b) no HuggingFace

In [ ]:
!pip install sae-lens transformer-lens datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.4/288.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.2/195.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('Sem GPU detectada. Processamento será muito lento.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

GPU: Tesla T4 (15.6 GB)
Device: cuda
PyTorch: 2.10.0+cu128


In [ ]:
from huggingface_hub import login
login()

---
# Fase 1: Setup Técnico

Carregamos o modelo Gemma 2 2B e o SAE pré-treinado (Gemma Scope) para a layer 13 com 16k features.

In [ ]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Carregando Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Modelo: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Carregando SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

Carregando Gemma 2 2B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Modelo: gemma-2-2b | Layers: 26 | d_model: 2304

Carregando SAE: layer_13/width_16k/canonical...


layer_13/width_16k/average_l0_84/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

SAE: 16384 features | d_in: 2304 | hook: blocks.13.hook_resid_post


/tmp/ipython-input-3908203106.py:17: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


In [ ]:
hook_name = HOOK_NAME

textos_teste = [
    ("PT", "O Brasil é o maior país da América do Sul e tem uma população diversa."),
    ("PT", "As meninas correram para a escola porque estavam atrasadas."),
    ("EN", "Brazil is the largest country in South America with a diverse population."),
    ("EN", "The girls ran to school because they were late."),
]

print("=== Teste Rápido ===")
print()
for lang, texto in textos_teste:
    tokens = model.to_tokens(texto)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=lambda name: name == hook_name)
        acts = cache[hook_name]
        feat_acts = sae.encode(acts)
    l0 = (feat_acts > 0).float().sum(dim=-1).mean().item()
    print(f'[{lang}] {texto[:55]}')
    print(f'     L0 médio: {l0:.1f} features/token')
    print()

print("Fase 1 completa! Modelo e SAE funcionando.")

=== Teste Rápido ===

[PT] O Brasil é o maior país da América do Sul e tem uma pop
     L0 médio: 437.3 features/token

[PT] As meninas correram para a escola porque estavam atrasa
     L0 médio: 556.7 features/token

[EN] Brazil is the largest country in South America with a d
     L0 médio: 510.6 features/token

[EN] The girls ran to school because they were late.
     L0 médio: 639.3 features/token

Fase 1 completa! Modelo e SAE funcionando.


---
# Fase 2: Análise Piloto

Processamos ~1M tokens em PT e ~1M em EN (Wikipedia) para calcular o **Language Specificity Index (LSI)** de cada feature.

**LSI = (freq_PT - freq_EN) / (freq_PT + freq_EN)**

- LSI ≈ +1 → feature ativa quase só em PT
- LSI ≈ -1 → feature ativa quase só em EN
- LSI ≈ 0 → feature cross-lingual

In [ ]:
N_TOKENS = 1_000_000
SEQ_LEN = 256
BATCH_SIZE = 8

n_seqs = N_TOKENS // SEQ_LEN
n_batches = n_seqs // BATCH_SIZE

print(f"Tokens por idioma: {N_TOKENS:,}")
print(f"Sequências: {n_seqs:,} ({SEQ_LEN} tokens cada)")
print(f"Batches: {n_batches:,} (batch size {BATCH_SIZE})")

Tokens por idioma: 1,000,000
Sequências: 3,906 (256 tokens cada)
Batches: 488 (batch size 8)


In [ ]:
from datasets import load_dataset

tokenizer = model.tokenizer

def collect_tokens(dataset, n_tokens, desc="Tokenizando"):
    """Coleta e tokeniza textos até atingir n_tokens."""
    all_ids = []
    n_articles = 0
    for article in tqdm(dataset, desc=desc):
        ids = tokenizer.encode(article["text"], add_special_tokens=False)
        all_ids.extend(ids)
        n_articles += 1
        if len(all_ids) >= n_tokens:
            break
    all_ids = all_ids[:n_tokens]
    n_seqs = len(all_ids) // SEQ_LEN
    tokens = torch.tensor(all_ids[:n_seqs * SEQ_LEN], dtype=torch.long).reshape(n_seqs, SEQ_LEN)
    print(f"  {n_articles} artigos -> {tokens.numel():,} tokens ({tokens.shape[0]} seqs x {SEQ_LEN})")
    return tokens

print("Carregando Wikipedia PT...")
wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
tokens_pt = collect_tokens(wiki_pt, N_TOKENS, desc="PT")

print()
print("Carregando Wikipedia EN...")
wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
tokens_en = collect_tokens(wiki_en, N_TOKENS, desc="EN")

Carregando Wikipedia PT...


README.md: 0.00B [00:00, ?B/s]

PT: 0it [00:00, ?it/s]

  174 artigos -> 999,936 tokens (3906 seqs x 256)

Carregando Wikipedia EN...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

EN: 0it [00:00, ?it/s]

  141 artigos -> 999,936 tokens (3906 seqs x 256)


In [ ]:
@torch.no_grad()
def compute_feature_stats(model, sae, tokens, batch_size=BATCH_SIZE, desc=""):
    """Processa tokens pelo modelo e SAE, retornando estatísticas por feature."""
    n_features = sae.cfg.d_sae
    hook = HOOK_NAME
    counts = torch.zeros(n_features, device=device)
    sums = torch.zeros(n_features, device=device)
    maxvals = torch.zeros(n_features, device=device)
    total = 0

    n_batches = (len(tokens) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(tokens), batch_size), desc=desc, total=n_batches):
        batch = tokens[i:i+batch_size].to(device)
        _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
        acts = cache[hook]
        feat_acts = sae.encode(acts)

        active = feat_acts > 0
        counts += active.float().sum(dim=(0, 1))
        sums += feat_acts.sum(dim=(0, 1))
        maxvals = torch.max(maxvals, feat_acts.amax(dim=(0, 1)))
        total += batch.numel()

        del cache, acts, feat_acts, active
        if device == "cuda":
            torch.cuda.empty_cache()

    return {"counts": counts.cpu(), "sums": sums.cpu(), "max": maxvals.cpu(), "total_tokens": total}


print("Processando PT (Wikipedia)...")
stats_pt = compute_feature_stats(model, sae, tokens_pt, desc="PT")
print(f'  {stats_pt["total_tokens"]:,} tokens | {(stats_pt["counts"] > 0).sum().item():,} features ativas')

print()
print("Processando EN (Wikipedia)...")
stats_en = compute_feature_stats(model, sae, tokens_en, desc="EN")
print(f'  {stats_en["total_tokens"]:,} tokens | {(stats_en["counts"] > 0).sum().item():,} features ativas')

Processando PT (Wikipedia)...


PT:   0%|          | 0/489 [00:00<?, ?it/s]

  999,936 tokens | 15,675 features ativas

Processando EN (Wikipedia)...


EN:   0%|          | 0/489 [00:00<?, ?it/s]

  999,936 tokens | 16,112 features ativas


In [ ]:
freq_pt = stats_pt["counts"] / stats_pt["total_tokens"]
freq_en = stats_en["counts"] / stats_en["total_tokens"]

lsi = (freq_pt - freq_en) / (freq_pt + freq_en + 1e-10)

alive = (stats_pt["counts"] + stats_en["counts"]) > 0
MIN_ACTS = 100
active = (stats_pt["counts"] + stats_en["counts"]) >= MIN_ACTS

n_total = sae.cfg.d_sae
n_dead = (~alive).sum().item()
n_alive = alive.sum().item()
n_active = active.sum().item()

print("=" * 55)
print("ESTATÍSTICAS GERAIS")
print("=" * 55)
print(f"Total features:            {n_total:,}")
print(f"Features mortas:           {n_dead:,} ({n_dead/n_total*100:.1f}%)")
print(f"Features vivas:            {n_alive:,} ({n_alive/n_total*100:.1f}%)")
print(f"Features ativas (>={MIN_ACTS}):   {n_active:,} ({n_active/n_total*100:.1f}%)")

lsi_a = lsi[active]
print()
print("DISTRIBUIÇÃO DE LSI (features ativas)")
print(f"  Média:    {lsi_a.mean().item():.4f}")
print(f"  Mediana:  {lsi_a.median().item():.4f}")
print(f"  PT-predominantes (LSI > 0.3):   {(lsi_a > 0.3).sum().item()}")
print(f"  EN-predominantes (LSI < -0.3):  {(lsi_a < -0.3).sum().item()}")
print(f"  Cross-lingual (|LSI| < 0.1):    {(lsi_a.abs() < 0.1).sum().item()}")

lsi_ranked_pt = lsi.clone()
lsi_ranked_pt[~active] = -2
top_pt_vals, top_pt_idx = lsi_ranked_pt.topk(10)

lsi_ranked_en = lsi.clone()
lsi_ranked_en[~active] = 2
top_en_vals, top_en_idx = (-lsi_ranked_en).topk(10)

print()
print("=" * 75)
print("TOP-10 FEATURES PT-ESPECÍFICAS")
print("=" * 75)
fmt = '{:<4} {:<10} {:<10} {:<12} {:<12} {}'
print(fmt.format('#', 'Feature', 'LSI', 'Freq PT', 'Freq EN', 'Max PT'))
print("-" * 75)
for i in range(10):
    idx = top_pt_idx[i].item()
    print(fmt.format(
        i+1, idx,
        f'{lsi[idx].item():+.4f}',
        f'{freq_pt[idx].item():.6f}',
        f'{freq_en[idx].item():.6f}',
        f'{stats_pt["max"][idx].item():.2f}'
    ))

print()
print("=" * 75)
print("TOP-10 FEATURES EN-ESPECÍFICAS (controle)")
print("=" * 75)
print(fmt.format('#', 'Feature', 'LSI', 'Freq PT', 'Freq EN', 'Max EN'))
print("-" * 75)
for i in range(10):
    idx = top_en_idx[i].item()
    print(fmt.format(
        i+1, idx,
        f'{lsi[idx].item():+.4f}',
        f'{freq_pt[idx].item():.6f}',
        f'{freq_en[idx].item():.6f}',
        f'{stats_en["max"][idx].item():.2f}'
    ))

ESTATÍSTICAS GERAIS
Total features:            16,384
Features mortas:           216 (1.3%)
Features vivas:            16,168 (98.7%)
Features ativas (>=100):   13,532 (82.6%)

DISTRIBUIÇÃO DE LSI (features ativas)
  Média:    -0.3766
  Mediana:  -0.4252
  PT-predominantes (LSI > 0.3):   818
  EN-predominantes (LSI < -0.3):  7978
  Cross-lingual (|LSI| < 0.1):    1775

TOP-10 FEATURES PT-ESPECÍFICAS
#    Feature    LSI        Freq PT      Freq EN      Max PT
---------------------------------------------------------------------------
1    10496      +0.9984    0.013787     0.000011     21.42
2    1906       +0.9983    0.187967     0.000163     21.82
3    8718       +0.9975    0.010510     0.000013     38.60
4    16057      +0.9950    0.047372     0.000118     36.98
5    4584       +0.9942    0.027276     0.000079     31.15
6    15135      +0.9941    0.006033     0.000018     25.35
7    12649      +0.9928    0.035984     0.000130     12.89
8    2294       +0.9926    0.050683     0.000187

In [ ]:
@torch.no_grad()
def find_top_examples(model, sae, tokens, feature_indices,
                      n_examples=10, batch_size=BATCH_SIZE, max_batches=200):
    """Encontra os tokens com maior ativação para cada feature."""
    hook = HOOK_NAME
    feat_list = feature_indices.tolist()
    results = {f: [] for f in feat_list}
    tok = model.tokenizer

    actual_batches = min(max_batches, (len(tokens) + batch_size - 1) // batch_size)
    for i in tqdm(range(0, actual_batches * batch_size, batch_size),
                  desc="Buscando exemplos", total=actual_batches):
        if i >= len(tokens):
            break
        batch = tokens[i:i+batch_size].to(device)
        _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
        acts = cache[hook]
        feat_acts = sae.encode(acts)

        for f_idx in feat_list:
            vals = feat_acts[:, :, f_idx]
            for b in range(batch.shape[0]):
                max_val, max_pos = vals[b].max(dim=0)
                if max_val.item() > 0:
                    pos = max_pos.item()
                    start = max(0, pos - 8)
                    end = min(batch.shape[1], pos + 12)
                    ctx = tok.decode(batch[b, start:end].tolist())
                    token_str = tok.decode([batch[b, pos].item()])
                    results[f_idx].append({
                        "act": max_val.item(),
                        "token": token_str,
                        "context": ctx,
                    })

        del cache, acts, feat_acts
        if device == "cuda":
            torch.cuda.empty_cache()

    for f in results:
        results[f].sort(key=lambda x: x["act"], reverse=True)
        results[f] = results[f][:n_examples]
    return results


print("Buscando exemplos para top-10 features PT-específicas...")
examples_pt = find_top_examples(model, sae, tokens_pt, top_pt_idx)

print()
print("=" * 80)
print("EXEMPLOS ATIVADORES — TOP-10 FEATURES PT-ESPECÍFICAS")
print("=" * 80)

for rank, f_idx in enumerate(top_pt_idx.tolist()):
    print()
    print("-" * 70)
    print(f'Feature {f_idx} | LSI = {lsi[f_idx].item():+.4f} | Rank #{rank+1}')
    print("-" * 70)
    exs = examples_pt.get(f_idx, [])
    if not exs:
        print("  (sem exemplos)")
        continue
    for ex in exs[:5]:
        print(f'  [{ex["act"]:.2f}] ...{ex["context"]}...')

Buscando exemplos para top-10 features PT-específicas...


Buscando exemplos:   0%|          | 0/200 [00:00<?, ?it/s]


EXEMPLOS ATIVADORES — TOP-10 FEATURES PT-ESPECÍFICAS

----------------------------------------------------------------------
Feature 10496 | LSI = +0.9984 | Rank #1
----------------------------------------------------------------------
  [21.42] ...

 Economia 

A economia da Argentina é a terceira maior da América Latina, com uma alta qualidade...
  [20.83] ... Nas regiões do sul, os verões são mornos e invernos frios com fortes nevoadas...
  [19.79] ...cionais da sociedade, que a dominação é construída socialmente e que é injusta, e...
  [19.36] ... do mundo. Sua densidade populacional é de 227 habitantes por quilômetro quadrado....
  [19.31] ... Política 

 Governo 

A Argentina é uma república constitucional e uma democracia representativa. O governo...

----------------------------------------------------------------------
Feature 1906 | LSI = +0.9983 | Rank #2
----------------------------------------------------------------------
  [20.74] ... por lutas partidárias. Contudo, não

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de LSI
lsi_vals = lsi[active].numpy()
axes[0].hist(lsi_vals, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].axvline(0.3, color='red', linestyle=':', linewidth=1, label='LSI = +/- 0.3')
axes[0].axvline(-0.3, color='red', linestyle=':', linewidth=1)
axes[0].set_xlabel('Language Specificity Index (LSI)')
axes[0].set_ylabel('Contagem de features')
axes[0].set_title(f'Distribuição de LSI ({n_active:,} features ativas)')
axes[0].legend()

# Scatter: freq PT vs EN
fp = freq_pt[active].numpy()
fe = freq_en[active].numpy()
colors = lsi[active].numpy()
sc = axes[1].scatter(fe, fp, c=colors, cmap='RdBu', s=5, alpha=0.5, vmin=-1, vmax=1)
diag_max = max(fp.max(), fe.max())
axes[1].plot([0, diag_max], [0, diag_max], 'k--', linewidth=0.8, alpha=0.5)
axes[1].set_xlabel('Frequência EN')
axes[1].set_ylabel('Frequência PT')
axes[1].set_title('Frequência de Ativação: PT vs EN')
plt.colorbar(sc, ax=axes[1], label='LSI')

plt.tight_layout()
plt.savefig('fig1_lsi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Heatmap das top features PT
fig, ax = plt.subplots(figsize=(8, 6))
data = np.column_stack([freq_pt[top_pt_idx].numpy(), freq_en[top_pt_idx].numpy()])
im = ax.imshow(data, aspect='auto', cmap='YlOrRd')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Wikipedia PT', 'Wikipedia EN'])
ax.set_yticks(range(10))
labels = [f'F{idx} (LSI={lsi[idx].item():+.2f})' for idx in top_pt_idx.tolist()]
ax.set_yticklabels(labels)
ax.set_title('Top-10 Features PT-específicas: Frequência por Idioma')
for i in range(10):
    for j in range(2):
        color = 'white' if data[i, j] > data.max() * 0.6 else 'black'
        ax.text(j, i, f'{data[i, j]:.5f}', ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, label='Frequência de ativação')
plt.tight_layout()
plt.savefig('fig2_top_features_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Salvar resultados
results = {
    'lsi': lsi, 'freq_pt': freq_pt, 'freq_en': freq_en,
    'stats_pt': stats_pt, 'stats_en': stats_en,
    'top_pt_idx': top_pt_idx, 'top_en_idx': top_en_idx,
    'active_mask': active,
}
torch.save(results, 'pilot_results.pt')
print('Resultados salvos em pilot_results.pt')
print('Figuras: fig1_lsi_distribution.png, fig2_top_features_heatmap.png')

Resultados salvos em pilot_results.pt
Figuras: fig1_lsi_distribution.png, fig2_top_features_heatmap.png
